In [4]:
import pandas as pd
import mysql.connector
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Conexão com banco de dados
conn = mysql.connector.connect(
  host = '35.199.115.174',
  user = 'looqbox-challenge',
  password = 'looq-challenge',
  database = 'looqbox-challenge'
)

# Executar Queries originais fornecidas pelo cliente (sem alterações)
query_1 ="""
SELECT
  STORE_CODE
  ,STORE_NAME
  ,START_DATE
  ,END_DATE
  ,BUSINESS_NAME
  ,BUSINESS_CODE
FROM data_store_cad
"""
query_2 = """
SELECT
  STORE_CODE
  ,DATE
  ,SALES_VALUE
  ,SALES_QTY
FROM data_store_sales
WHERE DATE BETWEEN '2019-01-01' AND '2019-12-31'
"""
df_store = pd.read_sql(query_1, conn)
df_sales = pd.read_sql(query_2, conn)
conn.close()

# Processamento e Filtro de Datas via Pandas
df_sales['DATE'] = pd.to_datetime(df_sales['DATE'])
mask = (df_sales['DATE'] >= '2019-10-01') & (df_sales['DATE'] <= '2019-12-31')
df_sales_filtered = df_sales.loc[mask]

# Merge dos DataFrames com base no STORE_CODE
df_merged = pd.merge(df_sales_filtered, df_store, on='STORE_CODE')

# Agrupamento e cálculo do Ticket Médio (TM)
df_grouped = df_merged.groupby(['STORE_NAME', 'BUSINESS_NAME']).agg({
 'SALES_VALUE': 'sum',
 'SALES_QTY': 'sum'
}).reset_index()

# TM = Valor Total de Vendas / Quantidade Total de Vendas
df_grouped['TM'] = (df_grouped['SALES_VALUE'] / df_grouped['SALES_QTY']).round(2)

display(df_grouped[['STORE_NAME', 'BUSINESS_NAME', 'TM']])

,STORE_NAME,BUSINESS_NAME,TM
0,Bahia,Atacado,15.39
1,Bangkok,Posto,13.67
2,Belem,Proximidade,15.37
3,Berlin,Proximidade,15.39
4,Buenos Aires,Atacado,15.39
5,Chicago,Varejo,15.53
6,Dubai,Atacado,15.39
7,Hong Kong,Farma,26.35
8,London,Farma,28.99
9,Madri,Farma,29.03
